# AutoGluon.TimeSeries V1.4 Cheatsheet

This notebook contains the key code snippets from the AutoGluon Time Series cheatsheet.

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Installation

AutoGluon (GitHub) supports Python 3.9 to 3.12 and is available for Linux, macOS, and Windows. The fastest way to install AutoGluon is through the `uv` package manager.

```bash
# Install uv package manager (faster than pip)
# !python -m pip install -U uv

# Install AutoGluon
# !uv pip install autogluon

# Or install with pip
# !python -m pip install autogluon
```

## Preparing Data

AutoGluon can generate forecasts for datasets consisting of **multiple univariates** time series. Here we use the M4 Competition Daily dataset to demonstrate how to do forecasting with AutoGluon. 

The data typically requires two parts: **raw data** (time series) and **static features** (metadata).

In [11]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. Load the raw time series data (time series values)
# NOTE: You would replace 'm4_daily.csv' with your own dataset path.
raw_data = pd.read_csv("https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_daily_subset/train.csv")
print("Raw Data Head:")
print(raw_data.head())

# 2. Load the static features (metadata for each time series ID)
static_features= pd.read_csv("https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_daily_subset/metadata.csv")
print("\nStatic Features Head:")
print(static_features.head())

Raw Data Head:
  item_id   timestamp  target
0   D1737  1995-05-23  1900.0
1   D1737  1995-05-24  1877.0
2   D1737  1995-05-25  1873.0
3   D1737  1995-05-26  1859.0
4   D1737  1995-05-27  1876.0

Static Features Head:
  item_id    domain
0   D1737  Industry
1   D1843  Industry
2   D2246   Finance
3    D909     Micro
4   D1345     Micro


In [12]:
raw_data['item_id'].unique()

array(['D1737', 'D1843', 'D2246', 'D909', 'D1345', 'D464', 'D3833',
       'D1310', 'D692', 'D1642', 'D2695', 'D1352', 'D3521', 'D569',
       'D832', 'D1186', 'D512', 'D3631', 'D3085', 'D2926', 'D458',
       'D2315', 'D271', 'D180', 'D2125', 'D3840', 'D189', 'D185', 'D1335',
       'D2471', 'D3335', 'D306', 'D97', 'D3306', 'D2631', 'D565', 'D1558',
       'D599', 'D167', 'D2078', 'D4036', 'D1942', 'D1069', 'D3878',
       'D1757', 'D151', 'D2563', 'D1107', 'D1722', 'D462', 'D2391',
       'D3259', 'D2501', 'D2537', 'D1362', 'D2551', 'D1755', 'D545',
       'D4132', 'D1815', 'D1972', 'D889', 'D1733', 'D1565', 'D539',
       'D2257', 'D158', 'D3415', 'D644', 'D882', 'D3958', 'D319', 'D2677',
       'D2513', 'D275', 'D3037', 'D1433', 'D656', 'D2267', 'D2045', 'D85',
       'D433', 'D970', 'D3053', 'D2410', 'D3086', 'D3124', 'D4023',
       'D2358', 'D3065', 'D409', 'D979', 'D2883', 'D2370', 'D721', 'D255',
       'D1922', 'D2321', 'D2104', 'D2345'], dtype=object)

### Convert Raw Data into a TimeSeriesDataFrame

Convert your raw data into the format required by AutoGluon: `TimeSeriesDataFrame`.

In [13]:
# from autogluon.timeseries import TimeSeriesDataFrame # Already imported above
train_data = TimeSeriesDataFrame(
    raw_data,
    id_column="item_id",
    timestamp_column="timestamp",
    static_features=static_features,  # Optional metadata/covariates
)

print("Processed Training Data:")
print(train_data.head())

Processed Training Data:
                    target
item_id timestamp         
D1737   1995-05-23  1900.0
        1995-05-24  1877.0
        1995-05-25  1873.0
        1995-05-26  1859.0
        1995-05-27  1876.0


## Training

Train models to forecast the values in the column `target` $30$ steps into the future.

In [14]:
# from autogluon.timeseries import TimeSeriesPredictor # Already imported above

predictor = TimeSeriesPredictor(
    target="target",
    prediction_length=30, # The number of time steps into the future to forecast
    # Optional: Additional covariates that are known in the future
    # known_covariates_names=['weekday', 'month'],
).fit(
    train_data,
    # Presets control the model quality and training time
    presets="medium_quality",
    # Other options: tuning metric, time limit, hyperparamter adjustment, etc.
    # eval_metric="MAPE",
    # time_limit=600,
    # verbosity=2
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/sergio/code/autogluon/AutogluonModels/ag-20251002_203853'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #32~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Sep  2 14:21:04 UTC 2
CPU Count:          12
GPU Count:          1
Memory Avail:       55.16 GB / 62.70 GB (88.0%)
Disk Space Avail:   115.43 GB / 217.97 GB (53.0%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 30,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'target',
 'verbosity': 2}

Inferred time series frequency: 'D'
Provided train_data has 2

## Predicting

Forecast `prediction_length` steps into the future starting from the end of each time series in `train_data`.

In [15]:
predictions = predictor.predict(
    train_data,
    # If you used known_covariates_names during training, pass them here:
    # known_covariates=known_covariates,
)

print("Forecasted Predictions Head:")
print(predictions.head())

# AutoGluon generates probabilistic forecasts that include:
# - mean_forecast - expected value of the time series
# - quantile_forecast - range of possible outcomes

# Predict on a new, unseen dataset
# predictions_test = predictor.predict_test_data(
#     test_data,
#     model_names=['DeepAR', 'ETS'], # Optional: specify which models to use
# )

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Forecasted Predictions Head:
                           mean          0.1          0.2          0.3  \
item_id timestamp                                                        
D1737   1997-05-28  1571.999756  1524.461426  1541.246338  1552.903564   
        1997-05-29  1572.034058  1509.016357  1532.508545  1546.536133   
        1997-05-30  1573.650879  1499.097168  1527.406494  1544.012695   
        1997-05-31  1573.254150  1490.471436  1522.300903  1540.580688   
        1997-06-01  1573.845703  1483.431885  1519.159668  1538.378418   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
D1737   1997-05-28  1563.825928  1571.999756  1579.561279  1587.682861   
        1997-05-29  1560.618408  1572.034058  1581.849121  1593.685547   
        1997-05-30  1559.964478  1573.650879  1585.581543  1599.321655   
        1997-05-31  1558.227051  1573.254150  1586.711182  1601.817383   
        

## Model Understanding

Understand the contribution of each model.

In [16]:
leaderboard = predictor.leaderboard()
print(leaderboard)

                       model  score_val  pred_time_val  fit_time_marginal  \
0           WeightedEnsemble  -0.032063       6.221011           1.344606   
1        Chronos[bolt_small]  -0.032268       6.080906           2.184705   
2  TemporalFusionTransformer  -0.032618       0.140105         163.687979   
3           RecursiveTabular  -0.033355       0.515338          20.349337   
4                      Theta  -0.034363       4.679522           0.106843   
5                      Naive  -0.034372       1.817529           0.079754   
6                        ETS  -0.034410      16.179974           0.069443   
7              SeasonalNaive  -0.037030       0.122280           0.106668   
8              DirectTabular  -0.038783       0.718130          83.817362   

   fit_order  
0          9  
1          7  
2          8  
3          3  
4          6  
5          1  
6          5  
7          2  
8          4  
